In [1]:
import requests as rq
import pandas as pd

from bs4 import BeautifulSoup

In [ ]:
url = 'https://www.worldometers.info/coronavirus/#countries'
response = rq.get(url)

In [ ]:
soup = BeautifulSoup(response.content, 'html.parser')

In [ ]:
#Tìm bảng dữ liệu của ngày mới nhất (13/4/2024)
rows = soup.find('table', id='main_table_countries_today').find_all('tr', class_=None)

In [ ]:
#Lấy header
headers = []
header_row = rows[0] #lấy phần tử đầu tiên của rows chứa header của bảng
for hd in header_row.find_all('th'):
    text = " ".join(hd.get_text(separator=' ').split())
    headers.append(text)
headers = [h.replace(" / ", "/").replace("/ ", "/") for h in headers]

In [ ]:
#Lấy dữ liệu trong bảng
rows.pop(0) #loại bỏ phần tử đầu tiên của rows chứa header của bảng
Data = []
for row in rows:
    data_row = []
    for td in row.find_all('td'):
        text = td.get_text(strip=True)
        data_row.append(text)
    Data.append(data_row)

In [ ]:
#Chuyển dữ liệu và header thành dataframe
df = pd.DataFrame(Data, columns=headers)

#Chuyển các dữ liệu dạng số từ string sang int, thay thế các giá trị null và NaN bằng giá trị 0
cols = df.columns[2:]
for col in cols:
    df[col] = pd.to_numeric(
        df[col].str.replace(',', ''), 
        errors='coerce'
    ).fillna(0).astype(int)

#Xuất re file csv để kiểm tra
df.to_csv("Data/Covid_19.csv", index=False)

    DO CASE CHỈ GHI NHẬN ĐẾN 12/4/2024, ĐỂ TÍNH "NEW CASES" SẼ LẤY DỮ LIỆU NGÀY 12/4/2024 TRONG BẢNG YESTERDAY

In [ ]:
#Lấy dữ liệu trong bảng, tận dụng header lấy từ bảng trước
rows_yesterday = soup.find('table', id='main_table_countries_yesterday').find_all('tr', class_=None)
rows_yesterday.pop(0)
Data = []
for row in rows_yesterday:
    data_row = []
    for td in row.find_all('td'):
        text = td.get_text(strip=True)
        data_row.append(text)
    Data.append(data_row)

In [ ]:
#Chuyển dữ liệu và header thành dataframe
df_yesterday = pd.DataFrame(Data, columns=headers)

#Chuyển các dữ liệu dạng số từ string sang int, thay thế các giá trị null và NaN bằng giá trị 0
cols = df_yesterday.columns[2:]
for col in cols:
    df_yesterday[col] = pd.to_numeric(
        df_yesterday[col].str.replace(',', ''), 
        errors='coerce'
    ).fillna(0).astype(int)
    
#Xuất re file csv để kiểm tra
df_yesterday.to_csv("Data/Covid_19_yesterday.csv", index=False)

: 

In [ ]:
# 1.A
# Sử dụng dữ liệu trong bảng Now
df_sorted = df.sort_values(by='Total Cases', ascending=False) # Sắp xếp dữ liệu giảm dần theo 'Total Cases'
df_sorted[['Country, Other', 'Total Cases']].head(5) # Lấy 5 dòng đầu tiên chính là 5 quốc gia có số ca nhiễm nhiều nhất

,"Country, Other",Total Cases
0,USA,111820082
1,India,45035393
2,France,40138560
3,Germany,38828995
4,Brazil,38743918


In [ ]:
# 1.B
# Sử dụng dữ liệu trong bảng Yesterday
df_yesterday_sorted = df_yesterday.sort_values(by='New Cases', ascending=False) # Sắp xếp dữ liệu giảm dần theo 'New Cases'
df_yesterday_sorted[['Country, Other', 'New Cases']].head(1) # Lấy dòng đầu tiên chính là quốc gia có số ca nhiễm mới nhiều nhất

,"Country, Other",New Cases
27,Chile,1215


In [ ]:
# 1.C
# Sử dụng dữ liệu trong bảng Now
df['Recovery Rate'] = df['Total Recovered'] / df['Total Cases'] # Tạo cột 'Recovery Rate' có giá trị bằng 'Total Recovery' / 'Total Cases'
df_sorted = df.sort_values(by='Recovery Rate', ascending=False) # Sắp xếp dữ liệu giảm dần theo 'Recovery Rate'
df_sorted[['Country, Other', 'Recovery Rate']].head(3) # Lấy 3 dòng đầu tiên chính là 3 quốc gia có tỉ lệ số ca bình phục / tổng số ca nhiễm cao nhất

,"Country, Other",Recovery Rate
227,Vatican City,1.000000
222,Falkland Islands,1.000000
31,DPRK,0.999984


https://github.com/nnbaohuy-dev/Crawling_Data.git